# CodeAlpha - Unemployment Analysis with Python

This project analyzes unemployment trends in India using Python. The analysis covers data cleaning, monthly trends, state-wise comparison, COVID-19 period analysis, correlation, and key observations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.unemployment import (
    clean_unemployment_data,
    summarize_monthly_unemployment,
)

print("Libraries imported successfully!")

## 1. Load Dataset

In [ ]:
# Public copy of the CodeAlpha-style unemployment dataset
url = 'https://raw.githubusercontent.com/SaiyamTuteja/EMPLOYMENT_AND_UNEMPLOYMENT_DATA_ANALYSIS/master/Unemployment_Rate_upto_11_2020.csv'
df = pd.read_csv(url)
df.columns = df.columns.str.strip()

print('Dataset loaded successfully!')
print('Shape:', df.shape)
display(df.head())

## 2. Understand the Dataset

In [ ]:
print('Columns:')
print(df.columns.tolist())

print('\nData types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isnull().sum())

print('\nSummary statistics:')
display(df.describe(include='all'))

## 3. Data Cleaning and Preparation

In [ ]:
# Validate and clean a copy of the source data using reusable project logic.
df, data_quality_audit = clean_unemployment_data(df)
print('Data quality audit:')
display(pd.Series(data_quality_audit, name='Rows'))
print('Cleaned shape:', df.shape)
display(df.head())

## 4. Overall Unemployment Trend

In [ ]:
# Build one monthly row per month using region-level averages.
# The weighted estimate is inferred from the supplied employment and
# labour-participation estimates; it is not an official national rate.
monthly = summarize_monthly_unemployment(df)

plt.figure(figsize=(11, 5))
plt.plot(
    monthly["Month"],
    monthly["regional_mean_unemployment_pct"],
    marker="o",
    label="Unweighted mean of regional rates",
)
plt.plot(
    monthly["Month"],
    monthly["labour_force_weighted_unemployment_pct"],
    marker="s",
    label="Estimated labour-force-weighted rate",
)
plt.title("Monthly Unemployment Estimates Across Regions")
plt.xlabel("Month")
plt.ylabel("Unemployment Rate (%)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Monthly summary:")
display(monthly.round(2))

## 5. State-wise Unemployment Analysis

In [ ]:
state_avg = (df.groupby('Region')['Estimated Unemployment Rate (%)']
               .mean()
               .sort_values(ascending=False))

print('Top 10 regions by average unemployment rate:')
display(state_avg.head(10).to_frame('Average Unemployment Rate (%)'))

plt.figure(figsize=(10, 6))
state_avg.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Regions by Average Unemployment Rate')
plt.xlabel('Average Unemployment Rate (%)')
plt.ylabel('Region')
plt.tight_layout()
plt.show()

## 6. COVID-19 Period Analysis

In [ ]:
# The dataset covers 2020 monthly observations. We compare Jan-Mar with Apr-Oct
# as a simple before/after comparison around the period when COVID-19 restrictions began.
df['Period'] = np.where(df['Date'] < pd.Timestamp('2020-04-01'), 'Jan-Mar 2020', 'Apr-Oct 2020')

period_summary = df.groupby('Period')['Estimated Unemployment Rate (%)'].agg(['mean','min','max']).round(2)
display(period_summary)

plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x='Period', y='Estimated Unemployment Rate (%)')
plt.title('Unemployment Rate: Jan-Mar vs Apr-Oct 2020')
plt.xlabel('Period')
plt.ylabel('Unemployment Rate (%)')
plt.show()

## 7. Highest Monthly Unemployment

In [ ]:
peak_row = df.loc[df["Estimated Unemployment Rate (%)"].idxmax()]
print("Highest recorded regional unemployment observation in this dataset:")
print("Region:", peak_row["Region"])
print("Date:", peak_row["Date"].date())
print("Unemployment Rate (%):", round(peak_row["Estimated Unemployment Rate (%)"], 2))

monthly_peak = monthly.loc[
    monthly["labour_force_weighted_unemployment_pct"].idxmax()
]
print("\nMonth with the highest estimated labour-force-weighted unemployment:")
print("Month:", monthly_peak["Month"])
print(
    "Estimated weighted rate (%):",
    round(monthly_peak["labour_force_weighted_unemployment_pct"], 2),
)
print(
    "Unweighted regional mean (%):",
    round(monthly_peak["regional_mean_unemployment_pct"], 2),
)
print(
    "Note: the weighted rate is an estimate inferred from the supplied "
    "regional employment and participation figures."
)

## 8. Correlation Analysis

In [ ]:
corr_cols = [
    'Estimated Unemployment Rate (%)',
    'Estimated Employed',
    'Estimated Labour Participation Rate (%)'
]

plt.figure(figsize=(7, 5))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Between Employment Indicators')
plt.tight_layout()
plt.show()

display(df[corr_cols].corr())

## 9. Employment vs Unemployment

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x='Estimated Employed',
    y='Estimated Unemployment Rate (%)',
    hue='Period'
)
plt.title('Estimated Employment vs Unemployment Rate')
plt.xlabel('Estimated Employed')
plt.ylabel('Unemployment Rate (%)')
plt.tight_layout()
plt.show()

## 10. Key Insights

In [ ]:
pre_avg = period_summary.loc['Jan-Mar 2020', 'mean']
post_avg = period_summary.loc['Apr-Oct 2020', 'mean']
change = post_avg - pre_avg

print('Key observations:')
print(f'- Average unemployment rate for Jan-Mar 2020: {pre_avg:.2f}%')
print(f'- Average unemployment rate for Apr-Oct 2020: {post_avg:.2f}%')
print(f'- Change between the two periods: {change:+.2f} percentage points')
print('- Unemployment rates vary considerably across regions.')
print('- The monthly trend helps identify periods of unusually high unemployment.')
print('- Employment and labour participation indicators provide additional context for interpreting unemployment.')

## Conclusion

This project analyzed unemployment data in India using Python. The dataset was cleaned and prepared before performing exploratory analysis. Monthly trends, regional differences, COVID-19 period comparisons, and relationships between employment indicators were studied using charts and summary statistics. The analysis shows how Python can be used to identify important patterns in real-world economic data and communicate those findings clearly.